In [0]:
from pyspark.sql import functions as F

# Find the date range from appointments
appointments = spark.table("silver_appointments")

date_range = appointments.select(
    F.to_date("appointment_date").alias("date")
).agg(
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date")
).collect()[0]

min_date = date_range["min_date"]
max_date = date_range["max_date"]

# Create one row for every date
dim_date = (
    spark.sql(f"""
        SELECT explode(
            sequence(
                to_date('{min_date}'),
                to_date('{max_date}'),
                interval 1 day
            )
        ) AS date
    """)
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("week", F.weekofyear("date"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_name", F.date_format("date", "EEEE"))
)

display(dim_date)

date,date_key,year,quarter,month,month_name,week,day,day_name
2024-01-01,20240101,2024,1,1,January,1,1,Monday
2024-01-02,20240102,2024,1,1,January,1,2,Tuesday
2024-01-03,20240103,2024,1,1,January,1,3,Wednesday
2024-01-04,20240104,2024,1,1,January,1,4,Thursday
2024-01-05,20240105,2024,1,1,January,1,5,Friday
2024-01-06,20240106,2024,1,1,January,1,6,Saturday
2024-01-07,20240107,2024,1,1,January,1,7,Sunday
2024-01-08,20240108,2024,1,1,January,2,8,Monday
2024-01-09,20240109,2024,1,1,January,2,9,Tuesday
2024-01-10,20240110,2024,1,1,January,2,10,Wednesday


In [0]:
dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_date")

In [0]:
display(spark.table("dim_date"))

date,date_key,year,quarter,month,month_name,week,day,day_name
2024-01-01,20240101,2024,1,1,January,1,1,Monday
2024-01-02,20240102,2024,1,1,January,1,2,Tuesday
2024-01-03,20240103,2024,1,1,January,1,3,Wednesday
2024-01-04,20240104,2024,1,1,January,1,4,Thursday
2024-01-05,20240105,2024,1,1,January,1,5,Friday
2024-01-06,20240106,2024,1,1,January,1,6,Saturday
2024-01-07,20240107,2024,1,1,January,1,7,Sunday
2024-01-08,20240108,2024,1,1,January,2,8,Monday
2024-01-09,20240109,2024,1,1,January,2,9,Tuesday
2024-01-10,20240110,2024,1,1,January,2,10,Wednesday


In [0]:
from pyspark.sql import functions as F

patients = spark.table("silver_patients")

dim_patient = (
    patients
    .select(
        "patient_id",
        "first_name",
        "last_name",
        "date_of_birth",
        "gender",
        "blood_group",
        "email",
        "phone"
    )
    .dropDuplicates(["patient_id"])
)

display(dim_patient)

patient_id,first_name,last_name,date_of_birth,gender,blood_group,email,phone
12,Karan,Singh,1969-03-17,MALE,A+,patient12@example.com,917000000011
18,Arjun,Patel,1998-09-02,FEMALE,A-,patient18@example.com,917000000017
38,Kabir,Sharma,1980-10-10,MALE,A+,patient38@example.com,917000000037
67,Karan,Gupta,1978-05-30,FEMALE,O-,patient67@example.com,917000000066
70,Rohan,Sharma,1949-11-05,MALE,A+,patient70@example.com,917000000069
93,Kabir,Iyer,1995-03-28,FEMALE,AB+,patient93@example.com,917000000092
16,Kabir,Iyer,1946-04-07,FEMALE,AB+,patient16@example.com,917000000015
64,Karan,Reddy,1969-01-21,FEMALE,A-,patient64@example.com,917000000063
74,Arjun,Nair,1970-05-13,FEMALE,AB+,patient74@example.com,917000000073
94,Rohan,Gupta,1968-07-12,FEMALE,O+,patient94@example.com,917000000093


In [0]:
dim_patient.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_patient")

In [0]:
display(spark.table("dim_patient"))

patient_id,first_name,last_name,date_of_birth,gender,blood_group,email,phone
12,Karan,Singh,1969-03-17,MALE,A+,patient12@example.com,917000000011
18,Arjun,Patel,1998-09-02,FEMALE,A-,patient18@example.com,917000000017
38,Kabir,Sharma,1980-10-10,MALE,A+,patient38@example.com,917000000037
67,Karan,Gupta,1978-05-30,FEMALE,O-,patient67@example.com,917000000066
70,Rohan,Sharma,1949-11-05,MALE,A+,patient70@example.com,917000000069
93,Kabir,Iyer,1995-03-28,FEMALE,AB+,patient93@example.com,917000000092
16,Kabir,Iyer,1946-04-07,FEMALE,AB+,patient16@example.com,917000000015
64,Karan,Reddy,1969-01-21,FEMALE,A-,patient64@example.com,917000000063
74,Arjun,Nair,1970-05-13,FEMALE,AB+,patient74@example.com,917000000073
94,Rohan,Gupta,1968-07-12,FEMALE,O+,patient94@example.com,917000000093


In [0]:
from pyspark.sql import functions as F

doctors = spark.table("silver_doctors")

dim_doctor = (
    doctors
    .select(
        "doctor_id",
        "first_name",
        "last_name",
        "specialization",
        "qualification",
        "experience_years",
        "employment_status",
        "license_number"
    )
    .dropDuplicates(["doctor_id"])
    .withColumn("effective_date", F.current_date())
    .withColumn("end_date", F.lit("9999-12-31").cast("date"))
    .withColumn("is_current", F.lit(True))
)

display(dim_doctor)

doctor_id,first_name,last_name,specialization,qualification,experience_years,employment_status,license_number,effective_date,end_date,is_current
1,Ananya,Sharma,General Medicine,"MBBS, MD",12,Active,LIC-000001,2026-09-05,9999-12-31,true


In [0]:
dim_doctor.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_doctor")

In [0]:
display(spark.table("dim_doctor"))

doctor_id,first_name,last_name,specialization,qualification,experience_years,employment_status,license_number,effective_date,end_date,is_current
1,Ananya,Sharma,General Medicine,"MBBS, MD",12,Active,LIC-000001,2026-09-05,9999-12-31,true


In [0]:
from pyspark.sql import functions as F

# Department Dimension
dim_department = (
    spark.table("silver_departments")
    .select(
        "department_id",
        "department_name",
        "department_type",
        "location"
    )
    .dropDuplicates(["department_id"])
)

# Diagnosis Dimension
dim_diagnosis = (
    spark.table("silver_diagnoses")
    .select(
        "diagnosis_id",
        "diagnosis_code",
        "diagnosis_name",
        "disease_category",
        "chronicity",
        "severity"
    )
    .dropDuplicates(["diagnosis_id"])
)

# Treatment Dimension
dim_treatment = (
    spark.table("silver_treatments")
    .select(
        "treatment_id",
        "treatment_name",
        "treatment_category",
        "standard_cost"
    )
    .dropDuplicates(["treatment_id"])
)

# Medication Dimension
dim_medication = (
    spark.table("silver_medications")
    .select(
        "medication_id",
        "generic_name",
        "brand_name",
        "dosage_form",
        "strength",
        "manufacturer",
        "unit_price"
    )
    .dropDuplicates(["medication_id"])
)

# Insurance Dimension
dim_insurance = (
    spark.table("silver_insurance_providers")
    .select(
        "insurance_provider_id",
        "provider_name",
        "provider_type",
        "contact_email"
    )
    .dropDuplicates(["insurance_provider_id"])
)

print("All five dimensions created successfully.")

All five dimensions created successfully.


In [0]:
dimensions = {
    "dim_department": dim_department,
    "dim_diagnosis": dim_diagnosis,
    "dim_treatment": dim_treatment,
    "dim_medication": dim_medication,
    "dim_insurance": dim_insurance
}

for table_name, df in dimensions.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )
    print(f"Created: {table_name}")

Created: dim_department
Created: dim_diagnosis
Created: dim_treatment
Created: dim_medication
Created: dim_insurance


In [0]:
appointments = spark.table("silver_appointments")

fact_patient_visits = (
    appointments
    .withColumn(
        "date_key",
        F.date_format(
            F.to_date("appointment_date"),
            "yyyyMMdd"
        ).cast("int")
    )
    .select(
        "appointment_id",
        "patient_id",
        "doctor_id",
        "department_id",
        "date_key",
        "appointment_date",
        "appointment_status",
        "visit_type",
        "wait_time_minutes",
        "consultation_duration_minutes"
    )
    .dropDuplicates(["appointment_id"])
)

display(fact_patient_visits)

appointment_id,patient_id,doctor_id,department_id,date_key,appointment_date,appointment_status,visit_type,wait_time_minutes,consultation_duration_minutes
12,12,1,1,20240920,2024-09-20,SCHEDULED,SPECIALIST,37,19
18,18,1,1,20260626,2026-06-26,COMPLETED,EMERGENCY,9,12
38,38,1,1,20260601,2026-06-01,COMPLETED,FOLLOW-UP,77,19
67,67,1,1,20250522,2025-05-22,SCHEDULED,SPECIALIST,82,50
70,70,1,1,20240214,2024-02-14,COMPLETED,EMERGENCY,79,18
93,93,1,1,20240825,2024-08-25,COMPLETED,SPECIALIST,68,47
161,61,1,1,20241201,2024-12-01,SCHEDULED,SPECIALIST,57,27
186,86,1,1,20250128,2025-01-28,COMPLETED,FOLLOW-UP,74,38
190,90,1,1,20260204,2026-02-04,COMPLETED,SPECIALIST,51,24
218,18,1,1,20240503,2024-05-03,COMPLETED,SPECIALIST,14,25


In [0]:
fact_patient_visits = (
    fact_patient_visits
    .withColumn(
        "is_completed",
        F.when(
            F.upper(F.col("appointment_status")) == "COMPLETED",
            1
        ).otherwise(0)
    )
    .withColumn(
        "is_cancelled",
        F.when(
            F.upper(F.col("appointment_status")) == "CANCELLED",
            1
        ).otherwise(0)
    )
)

display(fact_patient_visits)

appointment_id,patient_id,doctor_id,department_id,date_key,appointment_date,appointment_status,visit_type,wait_time_minutes,consultation_duration_minutes,is_completed,is_cancelled
12,12,1,1,20240920,2024-09-20,SCHEDULED,SPECIALIST,37,19,0,0
18,18,1,1,20260626,2026-06-26,COMPLETED,EMERGENCY,9,12,1,0
38,38,1,1,20260601,2026-06-01,COMPLETED,FOLLOW-UP,77,19,1,0
67,67,1,1,20250522,2025-05-22,SCHEDULED,SPECIALIST,82,50,0,0
70,70,1,1,20240214,2024-02-14,COMPLETED,EMERGENCY,79,18,1,0
93,93,1,1,20240825,2024-08-25,COMPLETED,SPECIALIST,68,47,1,0
161,61,1,1,20241201,2024-12-01,SCHEDULED,SPECIALIST,57,27,0,0
186,86,1,1,20250128,2025-01-28,COMPLETED,FOLLOW-UP,74,38,1,0
190,90,1,1,20260204,2026-02-04,COMPLETED,SPECIALIST,51,24,1,0
218,18,1,1,20240503,2024-05-03,COMPLETED,SPECIALIST,14,25,1,0


In [0]:
fact_patient_visits.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_patient_visits")

In [0]:
display(spark.table("fact_patient_visits"))

appointment_id,patient_id,doctor_id,department_id,date_key,appointment_date,appointment_status,visit_type,wait_time_minutes,consultation_duration_minutes,is_completed,is_cancelled
12,12,1,1,20240920,2024-09-20,SCHEDULED,SPECIALIST,37,19,0,0
18,18,1,1,20260626,2026-06-26,COMPLETED,EMERGENCY,9,12,1,0
38,38,1,1,20260601,2026-06-01,COMPLETED,FOLLOW-UP,77,19,1,0
67,67,1,1,20250522,2025-05-22,SCHEDULED,SPECIALIST,82,50,0,0
70,70,1,1,20240214,2024-02-14,COMPLETED,EMERGENCY,79,18,1,0
93,93,1,1,20240825,2024-08-25,COMPLETED,SPECIALIST,68,47,1,0
161,61,1,1,20241201,2024-12-01,SCHEDULED,SPECIALIST,57,27,0,0
186,86,1,1,20250128,2025-01-28,COMPLETED,FOLLOW-UP,74,38,1,0
190,90,1,1,20260204,2026-02-04,COMPLETED,SPECIALIST,51,24,1,0
218,18,1,1,20240503,2024-05-03,COMPLETED,SPECIALIST,14,25,1,0


In [0]:
from pyspark.sql import functions as F

appointment_treatments = spark.table("silver_appointment_treatments")

fact_treatments = (
    appointment_treatments
    .withColumn(
        "date_key",
        F.date_format(
            F.col("performed_date"),
            "yyyyMMdd"
        ).cast("int")
    )
    .select(
        "appointment_treatment_id",
        "appointment_id",
        "treatment_id",
        "date_key",
        "treatment_status",
        "quantity",
        "performed_date"
    )
    .dropDuplicates(["appointment_treatment_id"])
)

display(fact_treatments)

appointment_treatment_id,appointment_id,treatment_id,date_key,treatment_status,quantity,performed_date
14,14,1,20240626,Completed,3,2024-06-26
25,25,1,20240318,Completed,1,2024-03-18
39,39,1,20250318,Completed,3,2025-03-18
53,53,1,20260423,Recommended,2,2026-04-23
65,65,1,20250821,Completed,1,2025-08-21
97,97,1,20240813,Cancelled,3,2024-08-13
105,105,1,20240925,Completed,1,2024-09-25
181,181,1,20240722,Completed,3,2024-07-22
194,194,1,20240402,Started,3,2024-04-02
205,205,1,20250522,Completed,2,2025-05-22


In [0]:
fact_treatments.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_treatments")

In [0]:
display(spark.table("fact_treatments"))

appointment_treatment_id,appointment_id,treatment_id,date_key,treatment_status,quantity,performed_date
275,275,1,20240404,Completed,2,2024-04-04
38,38,1,20241120,Started,1,2024-11-20
218,218,1,20240504,Started,1,2024-05-04
261,261,1,20240418,Completed,3,2024-04-18
536,36,1,20241112,Completed,2,2024-11-12
70,70,1,20250220,Completed,1,2025-02-20
443,443,1,20240825,Recommended,3,2024-08-25
496,496,1,20251225,Started,1,2025-12-25
257,257,1,20240109,Cancelled,3,2024-01-09
581,81,1,20240903,Completed,1,2024-09-03


In [0]:
from pyspark.sql import functions as F

prescription_items = spark.table("silver_prescription_items")
prescriptions = spark.table("silver_prescriptions")

fact_prescriptions = (
    prescription_items.alias("pi")
    .join(
        prescriptions.alias("p"),
        F.col("pi.prescription_id") == F.col("p.prescription_id"),
        "inner"
    )
    .withColumn(
        "date_key",
        F.date_format(
            F.col("p.prescription_date"),
            "yyyyMMdd"
        ).cast("int")
    )
    .select(
        F.col("pi.prescription_item_id"),
        F.col("pi.prescription_id"),
        F.col("p.appointment_id"),
        F.col("p.patient_id"),
        F.col("p.prescribing_doctor_id").alias("doctor_id"),
        F.col("pi.medication_id"),
        F.col("date_key"),
        F.col("p.prescription_date"),
        F.col("p.status").alias("prescription_status"),
        F.col("p.start_date"),
        F.col("p.end_date"),
        F.col("pi.dosage"),
        F.col("pi.frequency"),
        F.col("pi.duration_days"),
        F.col("pi.quantity")
    )
    .dropDuplicates(["prescription_item_id"])
)

display(fact_prescriptions)

prescription_item_id,prescription_id,appointment_id,patient_id,doctor_id,medication_id,date_key,prescription_date,prescription_status,start_date,end_date,dosage,frequency,duration_days,quantity
452,152,152,52,1,1,20260127,2026-01-27,Active,2026-01-27,2026-02-07,500 mg,Once daily,12,1
600,300,300,100,1,1,20250828,2025-08-28,Completed,2025-08-28,2025-09-10,500 mg,Twice daily,24,8
161,161,161,61,1,1,20240428,2024-04-28,Cancelled,2024-04-28,2024-05-08,500 mg,As needed,20,10
218,218,218,18,1,1,20250422,2025-04-22,Cancelled,2025-04-22,2025-04-25,500 mg,Twice daily,4,3
439,139,139,39,1,1,20251223,2025-12-23,Completed,2025-12-23,2026-01-02,500 mg,As needed,24,44
581,281,281,81,1,1,20250530,2025-05-30,Completed,2025-05-30,2025-06-10,500 mg,Twice daily,10,59
93,93,93,93,1,1,20250529,2025-05-29,Completed,2025-05-29,2025-06-17,500 mg,Twice daily,16,17
257,257,257,57,1,1,20250405,2025-04-05,Completed,2025-04-05,2025-04-11,500 mg,Once daily,22,18
300,300,300,100,1,1,20250828,2025-08-28,Completed,2025-08-28,2025-09-10,500 mg,Twice daily,3,6
38,38,38,38,1,1,20250815,2025-08-15,Active,2025-08-15,2025-08-19,500 mg,As needed,14,20


In [0]:
fact_prescriptions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_prescriptions")

In [0]:
display(spark.table("fact_prescriptions"))

prescription_item_id,prescription_id,appointment_id,patient_id,doctor_id,medication_id,date_key,prescription_date,prescription_status,start_date,end_date,dosage,frequency,duration_days,quantity
452,152,152,52,1,1,20260127,2026-01-27,Active,2026-01-27,2026-02-07,500 mg,Once daily,12,1
600,300,300,100,1,1,20250828,2025-08-28,Completed,2025-08-28,2025-09-10,500 mg,Twice daily,24,8
161,161,161,61,1,1,20240428,2024-04-28,Cancelled,2024-04-28,2024-05-08,500 mg,As needed,20,10
218,218,218,18,1,1,20250422,2025-04-22,Cancelled,2025-04-22,2025-04-25,500 mg,Twice daily,4,3
439,139,139,39,1,1,20251223,2025-12-23,Completed,2025-12-23,2026-01-02,500 mg,As needed,24,44
581,281,281,81,1,1,20250530,2025-05-30,Completed,2025-05-30,2025-06-10,500 mg,Twice daily,10,59
93,93,93,93,1,1,20250529,2025-05-29,Completed,2025-05-29,2025-06-17,500 mg,Twice daily,16,17
257,257,257,57,1,1,20250405,2025-04-05,Completed,2025-04-05,2025-04-11,500 mg,Once daily,22,18
300,300,300,100,1,1,20250828,2025-08-28,Completed,2025-08-28,2025-09-10,500 mg,Twice daily,3,6
38,38,38,38,1,1,20250815,2025-08-15,Active,2025-08-15,2025-08-19,500 mg,As needed,14,20


In [0]:
from pyspark.sql import functions as F

lab_tests = spark.table("silver_lab_tests")
lab_results = spark.table("silver_lab_test_results")

fact_lab_tests = (
    lab_tests.alias("lt")
    .join(
        lab_results.alias("lr"),
        F.col("lt.lab_test_id") == F.col("lr.lab_test_id"),
        "left"
    )
    .withColumn(
        "date_key",
        F.date_format(
            F.col("lt.ordered_date"),
            "yyyyMMdd"
        ).cast("int")
    )
    .select(
        F.col("lt.lab_test_id"),
        F.col("lt.appointment_id"),
        F.col("lt.patient_id"),
        F.col("date_key"),
        F.col("lt.test_name"),
        F.col("lt.test_category"),
        F.col("lt.ordered_date"),
        F.col("lt.status").alias("test_status"),
        F.col("lt.lab_cost"),
        F.col("lr.lab_result_id"),
        F.col("lr.result_value"),
        F.col("lr.result_unit"),
        F.col("lr.reference_range"),
        F.col("lr.result_status"),
        F.col("lr.result_date")
    )
    .dropDuplicates(["lab_test_id"])
)

display(fact_lab_tests)

lab_test_id,appointment_id,patient_id,date_key,test_name,test_category,ordered_date,test_status,lab_cost,lab_result_id,result_value,result_unit,reference_range,result_status,result_date
190,190,90,20260506,HbA1c,Hematology,2026-05-06,Completed,840.03,190,49.85,mmol/L,20-40,Low,2024-10-28
12,12,12,20241207,HbA1c,Cardiology,2024-12-07,Completed,2795.41,12,29.42,IU/L,20-40,Normal,2026-03-24
317,317,17,20240910,HbA1c,Hematology,2024-09-10,Completed,1872.74,317,31.77,mg/dL,20-40,Normal,2025-07-28
343,343,43,20251015,Blood Glucose,Hematology,2025-10-15,Completed,4656.51,343,71.11,mg/dL,60-80,Low,2024-05-09
161,161,61,20260330,TSH,Hematology,2026-03-30,Ordered,3807.92,161,55.61,mmol/L,40-60,Normal,2024-04-24
186,186,86,20240710,TSH,Biochemistry,2024-07-10,Completed,2704.21,186,36.72,mmol/L,40-60,Normal,2024-09-15
273,273,73,20241222,TSH,Hematology,2024-12-22,Completed,1791.41,273,39.94,IU/L,40-60,Normal,2025-01-15
295,295,95,20251120,HbA1c,Cardiology,2025-11-20,Completed,2993.02,295,34.34,mg/dL,20-40,Normal,2024-08-14
93,93,93,20240310,CBC,Cardiology,2024-03-10,Completed,1225.91,93,29.67,IU/L,60-80,Normal,2026-01-25
70,70,70,20251127,CBC,Biochemistry,2025-11-27,Ordered,1570.05,70,34.56,mmol/L,40-60,Normal,2025-02-24


In [0]:
fact_lab_tests.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_lab_tests")

In [0]:
from pyspark.sql import functions as F

billing = spark.table("silver_billing")

fact_billing = (
    billing
    .withColumn(
        "date_key",
        F.date_format(
            F.col("billing_date"),
            "yyyyMMdd"
        ).cast("int")
    )
    .select(
        "billing_id",
        "appointment_id",
        "patient_id",
        "doctor_id",
        "department_id",
        "patient_insurance_id",
        "date_key",
        "billing_date",
        "gross_amount",
        "discount_amount",
        "tax_amount",
        "total_amount",
        "insurance_amount",
        "patient_amount",
        "payment_status"
    )
    .dropDuplicates(["billing_id"])
)

display(fact_billing)

billing_id,appointment_id,patient_id,doctor_id,department_id,patient_insurance_id,date_key,billing_date,gross_amount,discount_amount,tax_amount,total_amount,insurance_amount,patient_amount,payment_status
15,15,15,1,1,15,20240607,2024-06-07,68203.53,4883.61,10002.27,73322.19,62323.86,10998.33,Partially Paid
23,23,23,1,1,23,20241221,2024-12-21,10943.51,562.4,1205.23,11586.34,8689.76,2896.58,Paid
34,34,34,1,1,34,20240927,2024-09-27,37318.82,5041.7,467.79,32744.91,26195.93,6548.98,Partially Paid
35,35,35,1,1,35,20260731,2026-07-31,53693.34,6712.01,4833.38,51814.71,31088.83,20725.88,Pending
84,84,84,1,1,84,20260728,2026-07-28,56864.44,5775.83,4489.05,55577.66,33346.6,22231.06,Pending
92,92,92,1,1,92,20250804,2025-08-04,37166.21,4985.5,5538.37,37719.08,28289.31,9429.77,Paid
96,96,96,1,1,96,20240406,2024-04-06,20497.25,2448.23,1794.86,19843.88,17859.49,1984.39,Paid
113,113,13,1,1,113,20240922,2024-09-22,7925.66,820.89,123.04,7227.81,4336.69,2891.12,Paid
140,140,40,1,1,20,20240425,2024-04-25,85335.62,1574.09,2204.12,85965.65,77369.08,8596.57,Partially Paid
148,148,48,1,1,28,20251230,2025-12-30,42507.33,6121.91,4359.44,40744.86,32595.89,8148.97,Partially Paid


In [0]:
fact_billing.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_billing")

In [0]:
display(spark.table("fact_billing"))

billing_id,appointment_id,patient_id,doctor_id,department_id,patient_insurance_id,date_key,billing_date,gross_amount,discount_amount,tax_amount,total_amount,insurance_amount,patient_amount,payment_status
439,439,39,1,1,null,20260706,2026-07-06,32002.45,4627.75,1048.8,28423.5,0.0,28423.5,Paid
18,18,18,1,1,18,20250412,2025-04-12,33415.89,4270.25,771.94,29917.58,25429.94,4487.64,Paid
263,263,63,1,1,23,20250825,2025-08-25,15317.58,1925.31,1898.49,15290.76,13761.68,1529.08,Partially Paid
475,475,75,1,1,null,20250630,2025-06-30,48162.18,6454.54,6351.06,48058.7,0.0,48058.7,Paid
317,317,17,1,1,77,20260708,2026-07-08,80886.42,5408.26,3724.05,79202.21,63361.77,15840.44,Pending
70,70,70,1,1,70,20260307,2026-03-07,52501.97,6601.57,2573.39,48473.79,29084.27,19389.52,Partially Paid
225,225,25,1,1,105,20240820,2024-08-20,15274.36,1815.57,1747.38,15206.17,13685.55,1520.62,Paid
434,434,34,1,1,null,20250413,2025-04-13,27250.61,2342.6,3072.36,27980.37,0.0,27980.37,Partially Paid
496,496,96,1,1,null,20241226,2024-12-26,99971.51,1259.72,548.34,99260.13,0.0,99260.13,Paid
280,280,80,1,1,40,20240222,2024-02-22,24704.8,3469.74,1765.1,23000.16,20700.14,2300.02,Pending


In [0]:
from pyspark.sql import functions as F

claims = spark.table("silver_insurance_claims")

fact_insurance_claims = (
    claims
    .withColumn(
        "date_key",
        F.date_format(
            F.col("claim_date"),
            "yyyyMMdd"
        ).cast("int")
    )
    .select(
        "claim_id",
        "billing_id",
        "patient_id",
        "patient_insurance_id",
        "date_key",
        "claim_number",
        "claim_date",
        "claim_amount",
        "approved_amount",
        "claim_status"
    )
    .dropDuplicates(["claim_id"])
)

display(fact_insurance_claims)

claim_id,billing_id,patient_id,patient_insurance_id,date_key,claim_number,claim_date,claim_amount,approved_amount,claim_status
18,18,18,18,20250412,CLM-000018,2025-04-12,25429.94,17800.96,Partially Approved
27,27,27,27,20250531,CLM-000027,2025-05-31,19517.01,17565.31,Rejected
55,55,55,55,20260521,CLM-000055,2026-05-21,9239.66,9239.66,Rejected
57,57,57,57,20250623,CLM-000057,2025-06-23,35800.04,35800.04,Submitted
73,73,73,73,20260729,CLM-000073,2026-07-29,74608.8,74608.8,Rejected
81,81,81,81,20240818,CLM-000081,2024-08-18,69011.66,48308.16,Partially Approved
114,114,14,114,20240812,CLM-000114,2024-08-12,16531.39,14878.25,Approved
159,159,59,39,20240926,CLM-000159,2024-09-26,56715.51,56715.51,Approved
176,176,76,56,20240922,CLM-000176,2024-09-22,73144.34,73144.34,Submitted
194,194,94,74,20241006,CLM-000194,2024-10-06,54501.34,54501.34,Approved


In [0]:
fact_insurance_claims.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_insurance_claims")

In [0]:
display(spark.table("fact_insurance_claims"))

claim_id,billing_id,patient_id,patient_insurance_id,date_key,claim_number,claim_date,claim_amount,approved_amount,claim_status
18,18,18,18,20250412,CLM-000018,2025-04-12,25429.94,17800.96,Partially Approved
161,161,61,41,20241202,CLM-000161,2024-12-02,19066.68,19066.68,Partially Approved
38,38,38,38,20241017,CLM-000038,2024-10-17,12646.54,8852.58,Approved
67,67,67,67,20260211,CLM-000067,2026-02-11,78553.68,78553.68,Approved
186,186,86,66,20260813,CLM-000186,2026-08-13,14388.02,11510.42,Approved
12,12,12,12,20240817,CLM-000012,2024-08-17,53398.18,37378.73,Partially Approved
190,190,90,70,20250506,CLM-000190,2025-05-06,29763.14,29763.14,Submitted
93,93,93,93,20251113,CLM-000093,2025-11-13,32598.47,29338.62,Approved
70,70,70,70,20260307,CLM-000070,2026-03-07,29084.27,23267.42,Approved
194,194,94,74,20241006,CLM-000194,2024-10-06,54501.34,54501.34,Approved


In [0]:
fact_visits = spark.table("fact_patient_visits")
dim_patient = spark.table("dim_patient")
dim_doctor = spark.table("dim_doctor")
dim_department = spark.table("dim_department")
dim_date = spark.table("dim_date")

star_patient_visits = (
    fact_visits.alias("f")
    .join(
        dim_patient.alias("p"),
        F.col("f.patient_id") == F.col("p.patient_id"),
        "left"
    )
    .join(
        dim_doctor.alias("d"),
        F.col("f.doctor_id") == F.col("d.doctor_id"),
        "left"
    )
    .join(
        dim_department.alias("dept"),
        F.col("f.department_id") == F.col("dept.department_id"),
        "left"
    )
    .join(
        dim_date.alias("dt"),
        F.col("f.date_key") == F.col("dt.date_key"),
        "left"
    )
    .select(
        F.col("f.appointment_id"),
        F.col("f.patient_id"),
        F.concat(
            F.col("p.first_name"),
            F.lit(" "),
            F.col("p.last_name")
        ).alias("patient_name"),
        F.col("f.doctor_id"),
        F.concat(
            F.col("d.first_name"),
            F.lit(" "),
            F.col("d.last_name")
        ).alias("doctor_name"),
        F.col("d.specialization"),
        F.col("f.department_id"),
        F.col("dept.department_name"),
        F.col("f.date_key"),
        F.col("dt.year"),
        F.col("dt.month"),
        F.col("dt.month_name"),
        F.col("f.appointment_date"),
        F.col("f.appointment_status"),
        F.col("f.visit_type"),
        F.col("f.wait_time_minutes"),
        F.col("f.consultation_duration_minutes"),
        F.col("f.is_completed"),
        F.col("f.is_cancelled")
    )
)

display(star_patient_visits)

appointment_id,patient_id,patient_name,doctor_id,doctor_name,specialization,department_id,department_name,date_key,year,month,month_name,appointment_date,appointment_status,visit_type,wait_time_minutes,consultation_duration_minutes,is_completed,is_cancelled
12,12,Karan Singh,1,Ananya Sharma,General Medicine,1,General Medicine,20240920,2024,9,September,2024-09-20,SCHEDULED,SPECIALIST,37,19,0,0
18,18,Arjun Patel,1,Ananya Sharma,General Medicine,1,General Medicine,20260626,2026,6,June,2026-06-26,COMPLETED,EMERGENCY,9,12,1,0
38,38,Kabir Sharma,1,Ananya Sharma,General Medicine,1,General Medicine,20260601,2026,6,June,2026-06-01,COMPLETED,FOLLOW-UP,77,19,1,0
67,67,Karan Gupta,1,Ananya Sharma,General Medicine,1,General Medicine,20250522,2025,5,May,2025-05-22,SCHEDULED,SPECIALIST,82,50,0,0
70,70,Rohan Sharma,1,Ananya Sharma,General Medicine,1,General Medicine,20240214,2024,2,February,2024-02-14,COMPLETED,EMERGENCY,79,18,1,0
93,93,Kabir Iyer,1,Ananya Sharma,General Medicine,1,General Medicine,20240825,2024,8,August,2024-08-25,COMPLETED,SPECIALIST,68,47,1,0
161,61,Karan Kumar,1,Ananya Sharma,General Medicine,1,General Medicine,20241201,2024,12,December,2024-12-01,SCHEDULED,SPECIALIST,57,27,0,0
186,86,Aditya Reddy,1,Ananya Sharma,General Medicine,1,General Medicine,20250128,2025,1,January,2025-01-28,COMPLETED,FOLLOW-UP,74,38,1,0
190,90,Kabir Gupta,1,Ananya Sharma,General Medicine,1,General Medicine,20260204,2026,2,February,2026-02-04,COMPLETED,SPECIALIST,51,24,1,0
218,18,Arjun Patel,1,Ananya Sharma,General Medicine,1,General Medicine,20240503,2024,5,May,2024-05-03,COMPLETED,SPECIALIST,14,25,1,0


In [0]:
print("Total fact records:", fact_visits.count())
print("Star Schema records:", star_patient_visits.count())

Total fact records: 500
Star Schema records: 500


In [0]:
print(
    "Missing patient dimension:",
    star_patient_visits.filter(F.col("patient_name").isNull()).count()
)

print(
    "Missing doctor dimension:",
    star_patient_visits.filter(F.col("doctor_name").isNull()).count()
)

print(
    "Missing department dimension:",
    star_patient_visits.filter(F.col("department_name").isNull()).count()
)

print(
    "Missing date dimension:",
    star_patient_visits.filter(F.col("year").isNull()).count()
)

Missing patient dimension: 0
Missing doctor dimension: 0
Missing department dimension: 0
Missing date dimension: 0


# Slowly Changing Dimensions (SCD)

Slowly Changing Dimensions (SCD) are techniques used in data warehousing to manage changes in dimension data over time.

In this project, three SCD techniques are demonstrated:

### SCD Type 1
- Updates the existing dimension record with the latest value.
- Previous values are not retained.
- Used for attributes where historical changes are not required.
- Example: Patient email or phone number.

### SCD Type 2
- Preserves the complete history of changes.
- A new record is created whenever a tracked attribute changes.
- Uses `effective_date`, `end_date`, and `is_current` to identify the active and historical records.
- Example: Doctor specialization or employment status.

### SCD Type 3
- Stores the current value and the previous value in the same record.
- Maintains limited history.
- Example: Current specialization and previous specialization of a doctor.

These techniques allow the Gold layer to maintain accurate historical information for healthcare analytics.

## SCD Type 1 — Patient Dimension

The Patient Dimension uses SCD Type 1.

When patient information such as email or phone number changes, the existing record is updated with the latest value. The previous value is not retained.

Delta Lake MERGE is used to perform insert and update operations efficiently.

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

source_patients = (
    spark.table("silver_patients")
    .select(
        "patient_id",
        "first_name",
        "last_name",
        "date_of_birth",
        "gender",
        "blood_group",
        "email",
        "phone"
    )
    .dropDuplicates(["patient_id"])
)

target_patients = DeltaTable.forName(
    spark,
    "dim_patient"
)

(
    target_patients.alias("target")
    .merge(
        source_patients.alias("source"),
        "target.patient_id = source.patient_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("SCD Type 1 MERGE completed successfully.")

SCD Type 1 MERGE completed successfully.


In [0]:
print(
    "Patient records in dimension:",
    spark.table("dim_patient").count()
)

print(
    "Unique patient IDs:",
    spark.table("dim_patient")
    .select("patient_id")
    .distinct()
    .count()
)

Patient records in dimension: 100
Unique patient IDs: 100


## SCD Type 2 — Doctor Dimension

The Doctor Dimension uses SCD Type 2 to preserve the historical
versions of doctor information.

When a tracked attribute changes, the existing record is closed
by updating its `end_date` and `is_current` flag. A new record is
then inserted with the updated information.

SCD Type 2 uses:

- `effective_date` — date from which the version is valid
- `end_date` — date until which the version is valid
- `is_current` — identifies the currently active version

This approach allows historical healthcare analytics to use the
doctor information that was valid at the time of a particular visit.

### SCD Type 2 — Change Detection

For SCD Type 2, the incoming doctor record is compared with the
currently active record in the dimension.

If a tracked attribute changes:

1. The current record is marked as historical.
2. Its `end_date` is updated.
3. `is_current` is changed to `false`.
4. A new version of the doctor is inserted.
5. The new version receives a new `effective_date`.
6. The new version is marked as the current record.

This preserves the complete history of changes.

In [0]:
from pyspark.sql import functions as F

# Current Doctor Dimension
current_doctors = (
    spark.table("dim_doctor")
    .filter(F.col("is_current") == True)
)

display(current_doctors)

doctor_id,first_name,last_name,specialization,qualification,experience_years,employment_status,license_number,effective_date,end_date,is_current
1,Ananya,Sharma,General Medicine,"MBBS, MD",12,Active,LIC-000001,2026-09-05,9999-12-31,true


### SCD Type 2 — Simulating an Incoming Change

To demonstrate historical tracking, a controlled incoming record is
created with a changed doctor specialization.

This is only a demonstration of the SCD Type 2 process and does not
modify the original OLTP source data.

In [0]:
incoming_doctors = (
    current_doctors
    .withColumn(
        "specialization",
        F.when(
            F.col("doctor_id") == 1,
            F.lit("Neurology")
        ).otherwise(F.col("specialization"))
    )
)

display(incoming_doctors)

doctor_id,first_name,last_name,specialization,qualification,experience_years,employment_status,license_number,effective_date,end_date,is_current
1,Ananya,Sharma,Neurology,"MBBS, MD",12,Active,LIC-000001,2026-09-05,9999-12-31,true


### SCD Type 2 — Close the Previous Version

When a tracked attribute changes, the currently active dimension
record is closed.

The old version remains in the dimension as historical data.

Its:
- `end_date` becomes today's date
- `is_current` becomes `false`

A new version will be inserted in the next step.


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

doctor_dimension = DeltaTable.forName(
    spark,
    "dim_doctor"
)

(
    doctor_dimension.alias("target")
    .merge(
        incoming_doctors.alias("source"),
        """
        target.doctor_id = source.doctor_id
        AND target.is_current = true
        AND target.specialization <> source.specialization
        """
    )
    .whenMatchedUpdate(
        set={
            "end_date": "current_date()",
            "is_current": "false"
        }
    )
    .execute()
)

print("Previous doctor version closed successfully.")

Previous doctor version closed successfully.


### SCD Type 2 — Insert New Version

After closing the previous version, the changed doctor record is
inserted as a new dimension record.

The new version:
- Contains the updated attribute
- Gets today's `effective_date`
- Uses `9999-12-31` as the open-ended `end_date`
- Is marked as the current version

Both the historical and current versions are retained.

In [0]:
new_doctor_version = (
    incoming_doctors
    .withColumn("effective_date", F.current_date())
    .withColumn(
        "end_date",
        F.lit("9999-12-31").cast("date")
    )
    .withColumn("is_current", F.lit(True))
)

new_doctor_version.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("dim_doctor")

print("New doctor version inserted successfully.")

New doctor version inserted successfully.


In [0]:
display(
    spark.table("dim_doctor")
    .filter(F.col("doctor_id") == 1)
    .orderBy("effective_date")
)

doctor_id,first_name,last_name,specialization,qualification,experience_years,employment_status,license_number,effective_date,end_date,is_current
1,Ananya,Sharma,General Medicine,"MBBS, MD",12,Active,LIC-000001,2026-09-05,2026-09-05,false


### SCD Type 2 — Validation

The Doctor Dimension should now contain multiple versions of the same
doctor when a tracked attribute changes.

The historical version has `is_current = false`, while the latest
version has `is_current = true`.

This allows historical analytics to retain the doctor information
that was valid during earlier periods.

In [0]:
doctor_history = spark.table("dim_doctor")

display(
    doctor_history
    .filter(F.col("doctor_id") == 1)
    .select(
        "doctor_id",
        "first_name",
        "last_name",
        "specialization",
        "effective_date",
        "end_date",
        "is_current"
    )
    .orderBy("effective_date")
)

doctor_id,first_name,last_name,specialization,effective_date,end_date,is_current
1,Ananya,Sharma,General Medicine,2026-09-05,2026-09-05,false


In [0]:
print(
    "Total versions for Doctor 1:",
    doctor_history.filter(F.col("doctor_id") == 1).count()
)

print(
    "Current version count:",
    doctor_history
    .filter(
        (F.col("doctor_id") == 1) &
        (F.col("is_current") == True)
    )
    .count()
)

Total versions for Doctor 1: 1
Current version count: 0


### SCD Type 2 — Correct New Version

The previous version has been closed successfully. The new current
version must now be created explicitly from the incoming source
attributes.

The new record will have:
- The updated specialization
- Today's effective date
- `9999-12-31` as the end date
- `is_current = true`

In [0]:
from pyspark.sql import functions as F

# Create the new current version explicitly
new_doctor_version = (
    incoming_doctors
    .select(
        "doctor_id",
        "first_name",
        "last_name",
        "specialization",
        "qualification",
        "experience_years",
        "employment_status",
        "license_number"
    )
    .withColumn("effective_date", F.current_date())
    .withColumn(
        "end_date",
        F.lit("9999-12-31").cast("date")
    )
    .withColumn("is_current", F.lit(True))
)

display(new_doctor_version)

doctor_id,first_name,last_name,specialization,qualification,experience_years,employment_status,license_number,effective_date,end_date,is_current


In [0]:
from pyspark.sql import functions as F

incoming_doctors = (
    spark.table("silver_doctors")
    .select(
        "doctor_id",
        "first_name",
        "last_name",
        "specialization",
        "qualification",
        "experience_years",
        "employment_status",
        "license_number"
    )
    .withColumn(
        "specialization",
        F.when(
            F.col("doctor_id") == 1,
            F.lit("Neurology")
        ).otherwise(F.col("specialization"))
    )
)

display(incoming_doctors)

doctor_id,first_name,last_name,specialization,qualification,experience_years,employment_status,license_number
1,Ananya,Sharma,Neurology,"MBBS, MD",12,Active,LIC-000001


In [0]:
new_doctor_version = (
    incoming_doctors
    .withColumn("effective_date", F.current_date())
    .withColumn(
        "end_date",
        F.lit("9999-12-31").cast("date")
    )
    .withColumn("is_current", F.lit(True))
)

new_doctor_version.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("dim_doctor")

print("New current doctor version inserted successfully.")

New current doctor version inserted successfully.


In [0]:
doctor_history = spark.table("dim_doctor")

display(
    doctor_history
    .filter(F.col("doctor_id") == 1)
    .select(
        "doctor_id",
        "first_name",
        "last_name",
        "specialization",
        "effective_date",
        "end_date",
        "is_current"
    )
    .orderBy("effective_date")
)

doctor_id,first_name,last_name,specialization,effective_date,end_date,is_current
1,Ananya,Sharma,General Medicine,2026-09-05,2026-09-05,false
1,Ananya,Sharma,Neurology,2026-09-05,9999-12-31,true


In [0]:
print(
    "Total versions for Doctor 1:",
    doctor_history.filter(F.col("doctor_id") == 1).count()
)

print(
    "Current version count:",
    doctor_history
    .filter(
        (F.col("doctor_id") == 1) &
        (F.col("is_current") == True)
    )
    .count()
)

Total versions for Doctor 1: 2
Current version count: 1


## SCD Type 3 — Doctor Specialization

SCD Type 3 maintains limited historical information by storing the
current and previous values of a tracked attribute in the same row.

In this project:

- `current_specialization` stores the latest specialization.
- `previous_specialization` stores the immediately previous specialization.

Unlike SCD Type 2, SCD Type 3 does not create a separate row for
each historical change.

In [0]:
from pyspark.sql import functions as F

doctor_scd3_source = (
    spark.table("silver_doctors")
    .select(
        "doctor_id",
        "first_name",
        "last_name",
        "specialization",
        "qualification",
        "experience_years",
        "employment_status",
        "license_number"
    )
    .dropDuplicates(["doctor_id"])
)

dim_doctor_scd3 = (
    doctor_scd3_source
    .withColumnRenamed(
        "specialization",
        "current_specialization"
    )
    .withColumn(
        "previous_specialization",
        F.lit(None).cast("string")
    )
)

display(dim_doctor_scd3)

doctor_id,first_name,last_name,current_specialization,qualification,experience_years,employment_status,license_number,previous_specialization
1,Ananya,Sharma,General Medicine,"MBBS, MD",12,Active,LIC-000001,null


In [0]:
dim_doctor_scd3.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_doctor_scd3")

print("SCD Type 3 dimension created successfully.")

SCD Type 3 dimension created successfully.


### SCD Type 3 — Updating Current and Previous Values

When the tracked specialization changes, the existing dimension row
is updated instead of creating a new row.

The old current value is moved to `previous_specialization`, while
the incoming value becomes `current_specialization`.

This demonstrates limited historical tracking within a single row.

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

doctor_scd3 = DeltaTable.forName(
    spark,
    "dim_doctor_scd3"
)

(
    doctor_scd3.alias("target")
    .merge(
        incoming_doctors.alias("source"),
        "target.doctor_id = source.doctor_id"
    )
    .whenMatchedUpdate(
        set={
            "previous_specialization": "target.current_specialization",
            "current_specialization": "source.specialization"
        }
    )
    .execute()
)

print("SCD Type 3 update completed successfully.")

SCD Type 3 update completed successfully.


In [0]:
display(
    spark.table("dim_doctor_scd3")
    .select(
        "doctor_id",
        "first_name",
        "last_name",
        "previous_specialization",
        "current_specialization"
    )
)

doctor_id,first_name,last_name,previous_specialization,current_specialization
1,Ananya,Sharma,General Medicine,Neurology


## Star Schema — Final Validation

The Gold layer follows a Star Schema design.

Dimension tables provide descriptive information, while fact tables
store measurable healthcare events.

### Dimension Tables

- `dim_patient`
- `dim_doctor`
- `dim_department`
- `dim_diagnosis`
- `dim_treatment`
- `dim_medication`
- `dim_insurance`
- `dim_date`

### Fact Tables

- `fact_patient_visits`
- `fact_treatments`
- `fact_prescriptions`
- `fact_lab_tests`
- `fact_billing`
- `fact_insurance_claims`

The fact tables contain foreign-key references to the dimensions,
allowing healthcare events to be analyzed by patient, doctor,
department, date, treatment, medication, and insurance.

In [0]:
# Gold Layer Table Validation

gold_tables = [
    "dim_patient",
    "dim_doctor",
    "dim_department",
    "dim_diagnosis",
    "dim_treatment",
    "dim_medication",
    "dim_insurance",
    "dim_date",
    "fact_patient_visits",
    "fact_treatments",
    "fact_prescriptions",
    "fact_lab_tests",
    "fact_billing",
    "fact_insurance_claims",
]

for table in gold_tables:
    count = spark.table(table).count()
    print(f"{table}: {count} records")

dim_patient: 100 records
dim_doctor: 2 records
dim_department: 1 records
dim_diagnosis: 1 records
dim_treatment: 1 records
dim_medication: 1 records
dim_insurance: 1 records
dim_date: 961 records
fact_patient_visits: 500 records
fact_treatments: 600 records
fact_prescriptions: 600 records
fact_lab_tests: 400 records
fact_billing: 500 records
fact_insurance_claims: 200 records


## Star Schema Analytical Query

The following query combines the patient visit fact table with
patient, doctor, department, and date dimensions.

This demonstrates how the Gold Star Schema supports analytical
reporting without directly querying the normalized OLTP tables.

In [0]:
star_analysis = (
    spark.table("fact_patient_visits").alias("f")
    .join(
        spark.table("dim_patient").alias("p"),
        F.col("f.patient_id") == F.col("p.patient_id"),
        "left"
    )
    .join(
        spark.table("dim_doctor").alias("d"),
        F.col("f.doctor_id") == F.col("d.doctor_id"),
        "left"
    )
    .join(
        spark.table("dim_department").alias("dept"),
        F.col("f.department_id") == F.col("dept.department_id"),
        "left"
    )
    .join(
        spark.table("dim_date").alias("dt"),
        F.col("f.date_key") == F.col("dt.date_key"),
        "left"
    )
    .groupBy(
        "dt.year",
        "dt.month",
        "dept.department_name",
        "d.specialization"
    )
    .agg(
        F.countDistinct("f.appointment_id").alias("total_visits"),
        F.sum("f.is_completed").alias("completed_visits"),
        F.sum("f.is_cancelled").alias("cancelled_visits"),
        F.round(
            F.avg("f.wait_time_minutes"), 2
        ).alias("average_wait_time"),
        F.round(
            F.avg("f.consultation_duration_minutes"), 2
        ).alias("average_consultation_duration")
    )
    .orderBy("year", "month")
)

display(star_analysis)

year,month,department_name,specialization,total_visits,completed_visits,cancelled_visits,average_wait_time,average_consultation_duration
2024,1,General Medicine,General Medicine,11,8,2,48.09,37.55
2024,1,General Medicine,Neurology,11,8,2,48.09,37.55
2024,2,General Medicine,Neurology,14,14,0,38.43,36.29
2024,2,General Medicine,General Medicine,14,14,0,38.43,36.29
2024,3,General Medicine,General Medicine,17,12,3,43.35,33.18
2024,3,General Medicine,Neurology,17,12,3,43.35,33.18
2024,4,General Medicine,General Medicine,16,9,3,50.19,38.25
2024,4,General Medicine,Neurology,16,9,3,50.19,38.25
2024,5,General Medicine,General Medicine,15,11,2,47.53,29.4
2024,5,General Medicine,Neurology,15,11,2,47.53,29.4
